# M1.6 Train my detector (YOLOv8n, imgsz 320 and 640)
Classes: `0 body` (Model 1), `1 lid` (Model 2). Starts from **public** `yolov8n.pt` (COCO), trained only on **my** data.

1. On the laptop: `python scripts/split.py` then `python scripts/make_colab_bundle.py`
2. Runtime -> Change runtime type -> **T4 GPU**
3. Run all cells, upload `colab_bundle.zip` when asked
4. Download `m1_results.zip` at the end and unzip it into the repo root

In [ ]:
!pip -q install ultralytics==8.3.* 
import json, os, shutil, glob
from google.colab import files
up = files.upload()            # choose colab_bundle.zip
!rm -rf /content/bundle && mkdir -p /content/bundle && unzip -q colab_bundle.zip -d /content/bundle
%cd /content/bundle
!cat data/splits/data.yaml
targets = json.load(open('docs/targets.json')) if os.path.exists('docs/targets.json') else {}
print('targets written before training:', targets)

In [ ]:
from ultralytics import YOLO
EPOCHS = 80
runs = {}
for sz in (320, 640):
    m = YOLO('yolov8n.pt')
    m.train(data='data/splits/data.yaml', imgsz=sz, epochs=EPOCHS, patience=20, batch=32,
            seed=0, deterministic=True, project='runs', name=f'y8n_{sz}', exist_ok=True, plots=True)
    runs[sz] = f'runs/y8n_{sz}/weights/best.pt'
runs

In [ ]:
# Evaluate both models on the held-out REAL test split
results = {}
for sz, w in runs.items():
    m = YOLO(w)
    r = m.val(data='data/splits/data.yaml', split='test', imgsz=sz, batch=1, plots=True,
              project='runs', name=f'test_{sz}', exist_ok=True)
    per_class = {m.names[c]: {'ap50': float(r.box.ap50[i]), 'ap50_95': float(r.box.ap[i])}
                 for i, c in enumerate(r.box.ap_class_index)}
    results[sz] = {'imgsz': sz, 'mAP50': float(r.box.map50), 'mAP50_95': float(r.box.map),
                   'precision': float(r.box.mp), 'recall': float(r.box.mr), 'per_class': per_class,
                   'confusion_matrix': r.confusion_matrix.matrix.astype(int).tolist(),
                   'confusion_labels': list(m.names.values()) + ['background']}
for sz, v in results.items():
    print(sz, {k: round(v[k], 3) for k in ('mAP50', 'mAP50_95', 'precision', 'recall')})

In [ ]:
# Compare with my targets and pick the model for the edge device (320 is faster; keep it if it meets the target)
def verdict(v):
    out = {}
    for k in ('mAP50', 'mAP50_95', 'precision', 'recall'):
        t = targets.get(k)
        out[k] = {'achieved': round(v[k], 3), 'target': t, 'met': (t is None) or v[k] >= t}
    return out
chosen = 320 if all(x['met'] for x in verdict(results[320]).values()) else 640
report = {'train': {'base_weights': 'yolov8n.pt (public COCO)', 'epochs': EPOCHS},
          'results': results, 'vs_target': {sz: verdict(v) for sz, v in results.items()}, 'chosen_imgsz': chosen}
os.makedirs('out/evidence/M1', exist_ok=True); os.makedirs('out/models', exist_ok=True)
json.dump(report, open('out/evidence/M1/train_eval.json', 'w'), indent=2)
shutil.copy(runs[chosen], 'out/models/best.pt')
shutil.copy(runs[320], 'out/models/best_320.pt'); shutil.copy(runs[640], 'out/models/best_640.pt')
for sz in (320, 640):
    for f in ('confusion_matrix.png', 'confusion_matrix_normalized.png', 'PR_curve.png'):
        p = f'runs/test_{sz}/{f}'
        if os.path.exists(p): shutil.copy(p, f'out/evidence/M1/test_{sz}_{f}')
    for f in ('results.png', 'results.csv'):
        p = f'runs/y8n_{sz}/{f}'
        if os.path.exists(p): shutil.copy(p, f'out/evidence/M1/train_{sz}_{f}')
print('chosen imgsz', chosen); print(json.dumps(report['vs_target'], indent=1))

In [ ]:
# 10 wrong detections (images + table) for the chosen model
!python scripts/wrong_detections.py --model out/models/best.pt --data data/splits --imgsz {chosen} --n 10 --out out/evidence/M1/wrong_detections
!cat out/evidence/M1/wrong_detections/wrong_detections.md

In [ ]:
!cd out && zip -qr ../m1_results.zip . && cd ..
files.download('m1_results.zip')